In [1]:
import os, sys, json, joblib
import numpy as np
import pandas as pd

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

MODEL_PATH = os.path.join(REPO_ROOT,"src", "app", "train","models", "modeloptuna.pkl")  # your saved pipeline
THRESH_JOBLIB = os.path.join(REPO_ROOT, "models", "threshold.joblib")  # optional
OUT_PATH = os.path.join(REPO_ROOT, "data", "predictions", "predictions_repurchase.csv")
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Model path:", MODEL_PATH)
print("Output path:", OUT_PATH)



Repo root: c:\Users\gabri\OneDrive\ProyectoFinalMLOps
Model path: c:\Users\gabri\OneDrive\ProyectoFinalMLOps\src\app\train\models\modeloptuna.pkl
Output path: c:\Users\gabri\OneDrive\ProyectoFinalMLOps\data\predictions\predictions_repurchase.csv


In [2]:
from src.app.train.etl import UserGenerator
from src.app.train.feature_engineer import FeatureEngineer


In [ ]:
ug = UserGenerator()
ug.run_etl()              
df_raw = ug.df.copy()

# Feature engineering (y_repurchase_30d)
fe = FeatureEngineer(df_raw)
df_features = fe.run()

assert pd.api.types.is_datetime64_any_dtype(df_features["InvoiceDate"])
display(df_features.head())
print(df_features.dtypes.head(12))


,InvoiceDate,Quantity,Revenue,UnitPrice,Country,CustomerID,n_past_invoices,prev_date,recency_days,spend_prior,qty_prior,avg_ticket_prior,avg_qty_per_invoice_prior,next_date,days_to_next,y_repurchase_30d
0,2010-07-12 14:57:00,319.0,711.79,2.890000,Iceland,12347,0.0,NaT,9999.0,0.00,0.0,0.000000,0.000000,2011-02-08 08:48:00,210.0,0
1,2011-02-08 08:48:00,277.0,584.91,3.101818,Iceland,12347,1.0,2010-07-12 14:57:00,210.0,711.79,319.0,711.790000,319.000000,2011-07-04 10:43:00,146.0,0
2,2011-07-04 10:43:00,483.0,636.25,2.595417,Iceland,12347,2.0,2011-02-08 08:48:00,146.0,1296.70,596.0,648.350000,298.000000,2011-07-12 15:52:00,8.0,1
3,2011-07-12 15:52:00,192.0,224.82,1.230909,Iceland,12347,3.0,2011-07-04 10:43:00,8.0,1932.95,1079.0,644.316667,359.666667,2011-09-06 13:01:00,55.0,0
4,2011-09-06 13:01:00,196.0,382.52,2.978889,Iceland,12347,4.0,2011-07-12 15:52:00,55.0,2157.77,1271.0,539.442500,317.750000,NaT,NaN,0


InvoiceDate         datetime64[ns]
Quantity                   float64
Revenue                    float64
UnitPrice                  float64
Country                     object
CustomerID                   int64
n_past_invoices            float64
prev_date           datetime64[ns]
recency_days               float64
spend_prior                float64
qty_prior                  float64
avg_ticket_prior           float64
dtype: object


In [ ]:
# Load pipeline
from joblib import load

pipe = load(MODEL_PATH)

pre = pipe.named_steps.get("preprocessor") or pipe.named_steps.get("pre")
if pre is None:
    raise RuntimeError("Couldn't find the preprocessor step in the pipeline (expected 'preprocessor' or 'pre').")

transformer_map = {name: cols for name, trans, cols in pre.transformers_}

num_cols = list(transformer_map.get("num", []))
cat_cols = list(transformer_map.get("cat", []))
feat_cols = num_cols + cat_cols

missing = [c for c in feat_cols if c not in df_features.columns]
if missing:
    print("Missing columns in df_features:", missing)
else:
    print("OK — all expected columns present.")

X_new = df_features.reindex(columns=feat_cols)
X_new.shape


OK — all expected columns present.


(7358, 10)

In [ ]:
# Definition of the threshold
from src.app.train.train_mlflow_advance import TrainOptuna
from sklearn.metrics import precision_recall_curve, f1_score

target_column = "y_repurchase_30d"  

trainer = TrainOptuna(
    df=df_features,
    numeric_features=num_cols,
    categorical_features=cat_cols,
    target_column=target_column,
    n_trials=1,                      
    optimization_metric="roc_auc",   
)

X_train, X_test, y_train, y_test = trainer.train_test_split_by_quantiles()

proba_test = pipe.predict_proba(X_test)[:, 1]

# Umbral utilizando F1 en test
prec, rec, thr = precision_recall_curve(y_test.astype(int), proba_test)
if len(thr) > 0:
    f1s = [f1_score(y_test, (proba_test >= t).astype(int)) for t in thr]
    t_star = float(thr[int(np.argmax(f1s))])
else:
    t_star = 0.5  

print("Chosen decision_threshold (F1-optimal on test):", t_star)


c:\Users\gabri\OneDrive\ProyectoFinalMLOps\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Rango total: 2010-01-12 08:26:00 → 2011-11-10 16:50:00 | cutoff: 2011-10-11 16:50:00
train_end: 2011-09-01 00:00:00
train: 3217 | test: 820
pos_rate train=0.417 | test=0.359
Chosen decision_threshold (F1-optimal on test): 0.4564975615701025


In [ ]:
# Predicciones (guardado de resultados)
proba = pipe.predict_proba(X_new)[:, 1]
yhat  = (proba >= t_star).astype(int)

pred = df_features.copy()
pred["p_repurchase_30d"] = proba
pred["repurchase_flag"]  = yhat

pred.to_csv(OUT_PATH, index=False)
print("✅ predictions_repurchase.csv written at:", OUT_PATH)
display(pred[["CustomerID", "InvoiceDate", "p_repurchase_30d", "repurchase_flag"]].head(10))


c:\Users\gabri\OneDrive\ProyectoFinalMLOps\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


✅ predictions_repurchase.csv written at: c:\Users\gabri\OneDrive\ProyectoFinalMLOps\data\predictions\predictions_repurchase.csv


,CustomerID,InvoiceDate,p_repurchase_30d,repurchase_flag
0,12347,2010-07-12 14:57:00,0.000003,0
1,12347,2011-02-08 08:48:00,0.308069,0
2,12347,2011-07-04 10:43:00,0.376730,0
3,12347,2011-07-12 15:52:00,0.454627,0
4,12347,2011-09-06 13:01:00,0.487123,1
5,12348,2011-05-04 10:47:00,0.000004,0
6,12350,2011-02-02 16:01:00,0.000002,0
7,12352,2011-01-03 14:57:00,0.000002,0
8,12352,2011-01-03 15:52:00,0.266780,0
9,12352,2011-03-11 14:37:00,0.277978,0
